This is a modified version of v1 that allows searching for datasets based on their tags (to fetch the 31 datasets that we ran our sample questions on...) and which also adds my NEON DB link so that the info fetched from database can be embedded and added to the NEON DB, as well as allowing for a search method to search for (using cosine similarity and after embedding the input question) for datasets that are most related to teh user question. This is just a prototype and I am planning to make one which is much more sophisticated (more features extracted from database to create stronger embeddings + context added by DSPy agent to user question to make the cosine similarity search even stronger - could  do something like translate ).

In [ ]:
!pip install requests psycopg[binary] pgvector openai azure-core
!pip install -U openai azure-core
!pip install umap-learn

In [ ]:
import os
from google.colab import userdata

# Azure OpenAI
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://o3miniapi.openai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = userdata.get('Azure')
os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"] = "text-embedding-3-small"

# Neon (optional for now)
os.environ["NEON_DATABASE_URL"] = userdata.get("Neon")  # put your Neon URL in Colab secrets



In [ ]:
from __future__ import annotations

import os
import time
import json
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import requests

from openai import AzureOpenAI

# Postgres / pgvector
# pip install psycopg[binary] pgvector
import psycopg
from pgvector.psycopg import register_vector


# ----------------------------
# 1) data.gouv.fr API fetching
# ----------------------------

DATASETS_URL = "https://www.data.gouv.fr/api/1/datasets/"

# openai.api_type = "azure"
# openai.api_base = os.environ["AZURE_OPENAI_ENDPOINT"]
# openai.api_key = os.environ["AZURE_OPENAI_API_KEY"]
# openai.api_version = "2024-02-01"


@dataclass
class DatasetRecord:
    id: str
    title: str
    description: str
    tags: List[str]

    def to_embedding_text(self) -> str:
        """
        Concatenate fields (excluding ID) into a single string for embeddings.
        You can tweak formatting freely.
        """
        tags_str = ", ".join(self.tags) if self.tags else ""
        title = (self.title or "").strip()
        desc = (self.description or "").strip()

        parts = []
        if title:
            parts.append(f"Title: {title}")
        if desc:
            parts.append(f"Description: {desc}")
        if tags_str:
            parts.append(f"Tags: {tags_str}")

        # Keep it stable & readable
        return "\n".join(parts).strip()



class DataGouvDatasetsClient:
    def __init__(self, base_url: str = DATASETS_URL, timeout_s: int = 60):
        self.base_url = base_url
        self.timeout_s = timeout_s
        self.session = requests.Session()
        self.session.headers.update({"Accept": "application/json"})

    def fetch_page(
        self,
        page: int = 1,
        page_size: int = 50,
        q: Optional[str] = None,
        tags: Optional[Sequence[str]] = None,  # NEW
    ) -> Dict[str, Any]:
        """
        GET /api/1/datasets/?page=...&page_size=...&q=...&tag=...&tag=...

        NOTE: 'tag' is documented as a string[] query param, meaning it can be repeated.
        """
        params: Dict[str, Any] = {"page": page, "page_size": page_size}
        if q:
            params["q"] = q
        if tags:
            # requests encodes list values as repeated query params: tag=a&tag=b
            params["tag"] = list(tags)

        resp = self.session.get(self.base_url, params=params, timeout=self.timeout_s)
        resp.raise_for_status()
        return resp.json()

    def iter_datasets(
        self,
        mode: str = "single_page",   # "single_page" or "all_pages"
        page: int = 1,
        page_size: int = 50,
        q: Optional[str] = None,
        tags: Optional[Sequence[str]] = None,      # NEW (server-side filter)
        hard_limit: Optional[int] = None,          # safety limit across all pages
        require_any_of_tags: Optional[Set[str]] = None,  # NEW (client-side OR verification)
    ) -> Iterable["DatasetRecord"]:
        """
        - single_page: fetch exactly `page`
        - all_pages: follow `next_page` until exhausted

        `tags` is passed to the API as repeated query params (tag=...).
        `require_any_of_tags` is an optional client-side OR filter for safety.
        """
        if mode not in ("single_page", "all_pages"):
            raise ValueError("mode must be 'single_page' or 'all_pages'")

        fetched = 0
        current_page = page
        next_url: Optional[str] = None

        while True:
            if next_url is None:
                payload = self.fetch_page(page=current_page, page_size=page_size, q=q, tags=tags)
            else:
                resp = self.session.get(next_url, timeout=self.timeout_s)
                resp.raise_for_status()
                payload = resp.json()

            data = payload.get("data", []) or []
            for item in data:
                rec = DatasetRecord(
                    id=str(item.get("id", "") or ""),
                    title=str(item.get("title", "") or ""),
                    description=str(item.get("description", "") or ""),
                    tags=list(item.get("tags", []) or []),
                )

                if not rec.id and not rec.title and not rec.description and not rec.tags:
                    continue

                # Optional client-side OR filter (guarantees correctness)
                if require_any_of_tags:
                    if not set(rec.tags).intersection(require_any_of_tags):
                        continue

                yield rec
                fetched += 1
                if hard_limit is not None and fetched >= hard_limit:
                    return

            if mode == "single_page":
                return

            next_url = payload.get("next_page") or None
            if not next_url:
                return

    def iter_datasets_with_any_tags(
        self,
        any_tags: Sequence[str],          # NEW: OR semantics
        mode: str = "all_pages",
        page: int = 1,
        page_size: int = 50,
        q: Optional[str] = None,
        hard_limit: Optional[int] = None,
    ) -> Iterable["DatasetRecord"]:
        """
        Guaranteed OR semantics across tags by querying once per tag and deduplicating.

        Example:
          any_tags=["tarifs-voyageurs", "gtfs", "mobilite"]
        """
        want: Set[str] = set(any_tags)
        seen_ids: Set[str] = set()
        emitted = 0

        for tag in want:
            for rec in self.iter_datasets(
                mode=mode,
                page=page,
                page_size=page_size,
                q=q,
                tags=[tag],                    # server-side filter per tag
                require_any_of_tags=want,      # safety verification (OR)
                hard_limit=None,               # we enforce hard_limit after dedupe
            ):
                if rec.id in seen_ids:
                    continue
                seen_ids.add(rec.id)
                yield rec
                emitted += 1
                if hard_limit is not None and emitted >= hard_limit:
                    return


# ----------------------------
# 2) Embeddings (Azure OpenAI)
# ----------------------------


class AzureEmbeddingClient:
    def __init__(
        self,
        azure_endpoint: str,
        api_key: str,
        deployment: str,                 # your Azure *deployment name*
        api_version: str = "2024-02-01", # embeddings API version
        request_batch_size: int = 64,
        max_retries: int = 6,
    ):
        self.deployment = deployment
        self.request_batch_size = request_batch_size
        self.max_retries = max_retries

        # NOTE: azure_endpoint (NOT endpoint)
        self.client = AzureOpenAI(
            azure_endpoint=azure_endpoint,
            api_key=api_key,
            api_version=api_version,
        )

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        all_embeddings: List[List[float]] = []
        total_tokens = 0
        for start in range(0, len(texts), self.request_batch_size):
            batch = texts[start : start + self.request_batch_size]
            attempt = 0
            while True:
                try:
                    resp = self.client.embeddings.create(
                        model=self.deployment,  # In Azure, model=<deployment_name>
                        input=batch,
                    )
                    total_tokens += resp.usage.total_tokens
                    # Keep order stable
                    items = sorted(resp.data, key=lambda x: x.index)
                    all_embeddings.extend([it.embedding for it in items])
                    break
                except Exception as e:
                    attempt += 1
                    if attempt > self.max_retries:
                        raise
                    sleep_s = min(2 ** attempt, 30)
                    print(f"[embed retry {attempt}/{self.max_retries}] {type(e).__name__}: {e} -> sleeping {sleep_s}s")
                    time.sleep(sleep_s)

        PRICE_PER_1M_TOKENS_USD = 0.02  # for text-embedding-3-small (OpenAI list price)
        cost_usd = total_tokens / 1_000_000 * PRICE_PER_1M_TOKENS_USD
        print("tokens:", total_tokens, "estimated USD:", cost_usd)

        return all_embeddings



# ----------------------------
# 3) Neon Postgres + pgvector
# ----------------------------

class PgvectorStore:
    def __init__(self, postgres_url: str):
        self.postgres_url = postgres_url

    def connect(self):
        conn = psycopg.connect(self.postgres_url)

        # Make sure vector type exists BEFORE register_vector()
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        conn.commit()

        register_vector(conn)
        return conn

    def ensure_schema(self, dim: int, table: str = "dataset_embeddings"):
        create_sql = f"""
        CREATE EXTENSION IF NOT EXISTS vector;

        CREATE TABLE IF NOT EXISTS {table} (
            dataset_id TEXT PRIMARY KEY,
            title TEXT,
            description TEXT,
            tags TEXT[],
            content TEXT NOT NULL,
            embedding VECTOR({dim}) NOT NULL,
            updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );

        CREATE INDEX IF NOT EXISTS {table}_embedding_cosine_idx
          ON {table} USING hnsw (embedding vector_cosine_ops);
        """
        with self.connect() as conn:
            with conn.cursor() as cur:
                cur.execute(create_sql)
            conn.commit()

    def upsert_rows(self, rows, table: str = "dataset_embeddings"):
        upsert_sql = f"""
        INSERT INTO {table} (dataset_id, title, description, tags, content, embedding)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (dataset_id) DO UPDATE SET
            title = EXCLUDED.title,
            description = EXCLUDED.description,
            tags = EXCLUDED.tags,
            content = EXCLUDED.content,
            embedding = EXCLUDED.embedding,
            updated_at = NOW();
        """
        with self.connect() as conn:
            with conn.cursor() as cur:
                cur.executemany(upsert_sql, rows)
            conn.commit()

    def search(
        self,
        query_embedding: List[float],
        k: int = 10,
        table: str = "dataset_embeddings",
        any_tags: Optional[Sequence[str]] = None,
    ):
        """
        Returns top-k rows by cosine distance (smaller = more similar).
        Optionally filter to rows that overlap any of the provided tags.
        """
        sql = f"""
        SELECT
            dataset_id,
            title,
            description,
            tags,
            content,
            (embedding <=> (%s::vector)) AS distance
        FROM {table}
        WHERE (%s::text[] IS NULL OR tags && %s::text[])
        ORDER BY embedding <=> (%s::vector)
        LIMIT %s;
        """
        tag_array = list(any_tags) if any_tags else None

        with self.connect() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (query_embedding, tag_array, tag_array, query_embedding, k))
                return cur.fetchall()


# ----------------------------
# 4) Orchestration / main
# ----------------------------

def run_pipeline(
    mode: str = "single_page",   # "single_page" or "all_pages"
    page: int = 1,
    page_size: int = 50,
    q: Optional[str] = None,
    hard_limit: Optional[int] = None,
    any_tags: Optional[Sequence[str]] = None,  # NEW
):
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY", "")
    azure_deployment = os.environ.get("AZURE_OPENAI_EMBED_DEPLOYMENT", "text-embedding-3-small")
    neon_postgres_url = os.environ.get("NEON_DATABASE_URL", "")

    if not azure_endpoint or not azure_api_key:
        raise RuntimeError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY env vars first.")

    dg = DataGouvDatasetsClient()

    # 1) fetch datasets (NEW: OR-tag mode)
    if any_tags:
        records = list(
            dg.iter_datasets_with_any_tags(
                any_tags=any_tags,
                mode="all_pages",       # OR-tag mode should generally scan all pages
                page=page,
                page_size=page_size,
                q=q,
                hard_limit=hard_limit,
            )
        )
    else:
        records = list(dg.iter_datasets(mode=mode, page=page, page_size=page_size, q=q, hard_limit=hard_limit))

    # 2) build embedding texts
    texts = [r.to_embedding_text() for r in records]

    # 3) embed
    emb_client = AzureEmbeddingClient(
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        deployment=os.environ.get("AZURE_OPENAI_EMBED_DEPLOYMENT", "text-embedding-3-small"),
    )

    embeddings = emb_client.embed_texts(texts)

    if len(embeddings) != len(records):
        raise RuntimeError(f"Embedding count mismatch: {len(embeddings)} != {len(records)}")

    dim = len(embeddings[0]) if embeddings else 0
    print(f"Fetched {len(records)} datasets, embedding_dim={dim}")

    # 4) upsert into pgvector (optional)
    if neon_postgres_url:
        store = PgvectorStore(neon_postgres_url)
        store.ensure_schema(dim=dim, table="dataset_embeddings")

        rows = []
        for rec, content, emb in zip(records, texts, embeddings):
            rows.append((rec.id, rec.title, rec.description, rec.tags, content, emb))

        store.upsert_rows(rows, table="dataset_embeddings")
        print(f"Upserted {len(rows)} rows into Neon.")
    else:
        print("NEON_DATABASE_URL not set yet — skipping DB write.")

    return records, embeddings

**Note about this next code block:**

You DO NOT have to run this code block every time you open this notebook, since really it's only function is to fill in the NEON DB, which it has already done. You should only run this next block if you want to:

1.   Upload all the dataset embeddings to NEON DB again (it will *likely* (but you should check) delete all the previous table entries)
2.   **Update** all the dataset embeddings to make sure that they are all up to date. (I have not actually added the functionality to update just yet - this will come once we decide to embed all 70,000 datasets, as of now, and as you can see from the below codeblock, we are only embedding the 31 datasets which have one of the following tags: tarif-voyageurs, autobus, and vie-etudiante.)


See my note on Feb 8 for a better explanation on this. I kinda rushed it here ...



In [ ]:
records, embeddings = run_pipeline(
    any_tags=["tarifs-voyageurs", "autobus", "vie-etudiante"],
    page_size=50,
    hard_limit=2000,   # optional
)

The following function is what will be used by the DSPy agent after it augments information to the user question.

(See v2 of this file for the updated code)

In [ ]:
def search_datasets(query: str, k: int = 5):
    emb_client = AzureEmbeddingClient(
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        deployment=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    )
    q_emb = emb_client.embed_texts([query])[0]

    store = PgvectorStore(os.environ["NEON_DATABASE_URL"])
    return store.search(q_emb, k=k)

In [ ]:
hits = search_datasets("données de transport en commun GTFS en France", k=5)
for dataset_id, title, desc, tags, content, dist in hits:
    print(dist, title, dataset_id)

tokens: 11 estimated USD: 2.2e-07
0.31368853355773263 Horaires théoriques et temps réel  (GTFS & GTFS-RT) du réseau Palmbus - Cannes Pays de Lérins 5c65bad88b4c4143d45c9d5c
0.3170108184526075 GTFS - PYBUS - Le réseau urbain de Parthenay 675465e5cb8ed4046fb125de
0.32127560992175463 Réseau de transport urbain de la Métropole Toulon Provence Méditerranée 5b6c1650634f4160a32d2a77
0.3218221643845137 Horaires des lignes du réseau transgironde 596067c4a3a7296407d6a7ce
0.33083065313557547 Horaires des lignes du réseau transgironde 5959231ea3a7291dcf9c8203


In [ ]:

import umap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE



# embeddings: List[List[float]]
E = np.array(embeddings, dtype=np.float32)

# PCA EMBEDDINGS

print("PCA EMBEDDINGS...\n")
# reduce to 2D
XY = PCA(n_components=2, random_state=0).fit_transform(E)

plt.figure(figsize=(8,6))
plt.scatter(XY[:,0], XY[:,1], s=8)
plt.title("Embeddings (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# t-SNE EMBEDDINGS

print("t-SNE EMBEDDINGS...\n")
XY = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=0).fit_transform(E)

plt.figure(figsize=(8,6))
plt.scatter(XY[:,0], XY[:,1], s=8)
plt.title("Embeddings (t-SNE 2D)")
plt.show()


# UMAP EMBEDDINGS

print("UMAP EMBEDDINGS...\n")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    n_jobs=-1   # use all cores
)
XY = reducer.fit_transform(E)

plt.figure(figsize=(8,6))
plt.scatter(XY[:,0], XY[:,1], s=8)
plt.title("Embeddings (UMAP 2D)")
colors = [len(r.tags) for r in records]  # or org, theme, etc.
plt.scatter(XY[:,0], XY[:,1], c=colors, s=8, cmap="viridis")
plt.colorbar(label="Number of tags")
plt.show()

# - All pages (be careful: lots of datasets)
# run_pipeline(mode="all_pages", page=1, page_size=100, hard_limit=None)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# records: List[DatasetRecord]
i = 0  # pick a dataset index
sims = cosine_similarity(E[i:i+1], E)[0]
top = sims.argsort()[-6:][::-1]  # itself + 5 neighbors

print("Query:", records[i].title)
print("---- Neighbors ----")
for j in top:
    print(f"{sims[j]:.3f} | {records[j].title}")


Query: Les lieux inspirants de l'Enseignement supérieur
---- Neighbors ----
1.000 | Les lieux inspirants de l'Enseignement supérieur
0.687 | Référents entrepreneuriat - PEPITE
0.588 | Dates des élections étudiantes aux conseils d'administration des CROUS et localisation des bureaux de vote
0.586 | Cartographie des cellules de lutte contre les violences sexistes et sexuelles
0.578 | Ensemble des logements proposés aux étudiants par le réseau des CROUS
0.567 | Ensemble des lieux de restauration des CROUS
